In [2]:
# Install once if needed
# !pip install pdfplumber pandas openpyxl pymupdf

import pdfplumber
import pandas as pd
import re
from pathlib import Path
import fitz
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

# =========================
# EDIT YOUR PHRASES HERE
# =========================
PHRASES = [
    "Stayed one night",
    "Savannah, Georgia",
    "party",
    "company",
    "pool table",
    "projector",
    "ping pong",
    "pool",
]

pdf_folder = Path(".")
results = []

# =========================
# SCAN PDF TEXT
# =========================
results = []

# only include original PDFs, skip any previously highlighted ones
pdf_files = [f for f in pdf_folder.glob("*.pdf") if not f.stem.endswith("_HIGHLIGHTED")]

for pdf_file in pdf_files:
    with pdfplumber.open(pdf_file) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if not text:
                continue

            lines = text.split("\n")

            for line_num, line in enumerate(lines, start=1):
                for phrase in PHRASES:
                    if re.search(re.escape(phrase), line, re.IGNORECASE):
                        # append " - Local Guest" if line is exactly "Savannah, Georgia"
                        line_text = line.strip()
                        if line_text.lower() == "savannah, georgia":
                            line_text += " - Local Guest"

                        results.append({
                            "file": pdf_file.name,
                            "page": page_num,
                            "line_number": line_num,
                            "phrase": phrase,
                            "line_text": line_text
                        })

df = pd.DataFrame(results)

# =========================
# SUMMARY TABLES
# =========================
phrase_summary = (
    df.groupby("phrase")
      .size()
      .reset_index(name="match_count")
      .sort_values("match_count", ascending=False)
)

location_summary = (
    df.groupby(["file", "page"])
      .size()
      .reset_index(name="matches_on_page")
      .sort_values(["file", "page"])
)

# =========================
# HIGHLIGHT MATCHES IN PDF
# =========================
for pdf_file in pdf_files:  # only highlight original PDFs
    doc = fitz.open(pdf_file)
    modified = False

    for page in doc:
        page_text = page.get_text("text")

        for phrase in PHRASES:
            pattern = re.compile(re.escape(phrase), re.IGNORECASE)

            for match in pattern.finditer(page_text):
                matched_text = match.group(0)
                text_instances = page.search_for(matched_text)

                for inst in text_instances:
                    highlight = page.add_highlight_annot(inst)
                    highlight.update()
                    modified = True

    if modified:
        output_name = pdf_file.stem + "_HIGHLIGHTED.pdf"
        doc.save(output_name, garbage=4, deflate=True)
    doc.close()

# =========================
# EXPORT EXCEL
# =========================
df.to_excel("airbnb_matches_detailed.xlsx", index=False)
phrase_summary.to_excel("airbnb_phrase_summary.xlsx", index=False)
location_summary.to_excel("airbnb_page_location_summary.xlsx", index=False)

# =========================
# GENERATE HTML REPORT
# =========================
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

html = f"""
<html>
<head>
<title>Airbnb Review Keyword Report</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 40px; }}
h1, h2 {{ color: #2c3e50; }}
table {{ border-collapse: collapse; width: 100%; margin-bottom: 30px; }}
th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; }}
th {{ background-color: #f2f2f2; }}
tr:nth-child(even) {{ background-color: #fafafa; }}
.summary-box {{
    padding: 15px;
    background: #f8f9fa;
    border: 1px solid #ddd;
    margin-bottom: 30px;
}}
</style>
</head>
<body>

<h1>Airbnb Review Keyword Report</h1>
<p><strong>Generated:</strong> {timestamp}</p>

<div class="summary-box">
<h2>Overview</h2>
<p><strong>Total Matches:</strong> {len(df)}</p>
<p><strong>Total PDFs Scanned:</strong> {len(pdf_files)}</p>
<p><strong>Phrases Searched:</strong> {", ".join(PHRASES)}</p>
</div>

<h2>Phrase Summary</h2>
{phrase_summary.to_html(index=False)}



<h2>Detailed Matches</h2>
{df.to_html(index=False)}

<h2>PDF Page Hotspots (Matches per Page)</h2>
{location_summary.to_html(index=False)}

</body>
</html>
"""

with open("airbnb_issue_report.html", "w", encoding="utf-8") as f:
    f.write(html)

# =========================
# DISPLAY IN NOTEBOOK
# =========================
print("HTML report saved as: airbnb_issue_report.html")
print("Open it in your browser → Print → Save as PDF")

display(phrase_summary)
display(location_summary)
display(df)

HTML report saved as: airbnb_issue_report.html
Open it in your browser → Print → Save as PDF


,phrase,match_count
0,pool,2


,file,page,matches_on_page
0,9240_garland_drive.pdf,1,1
1,9240_garland_drive.pdf,2,1


,file,page,line_number,phrase,line_text
0,9240_garland_drive.pdf,1,20,pool,As a whole property was great. The pool and ou...
1,9240_garland_drive.pdf,2,11,pool,"timely. The neighbors don’t bother at all, but..."
